# traineo1 Kaggle Clean

Notebook ini dibuat ulang supaya lebih sederhana dan lebih cocok untuk pemula.

Yang notebook ini lakukan:
- clone repo dulu
- bikin venv di `/kaggle/temp`
- install package seperlunya
- pakai dataset lokal dari Kaggle Input, jadi tidak download dataset lagi
- jalanin fine-tune konservatif

Catatan penting:
- Isi `REPO_URL` dan `REPO_BRANCH` dengan repo atau branch yang **sudah berisi perubahan konservatifmu**.
- Kalau branch yang di-clone belum punya file hybrid konservatif, notebook akan berhenti dengan error yang jelas.
- Setiap cell sengaja dibuat kecil dan fokus ke satu hal.


In [ ]:
# Cell 1: Repo config
REPO_URL = "https://github.com/<username>/F5-TTS.git"
REPO_BRANCH = "your-conservative-branch"
REPO_NAME = "gardenbunga"


In [ ]:
# Cell 2: Dataset config
DATASET_NAME = "datasetku"
DATASET_ROOT_RAW = "/kaggle/input/tts-indo"
CSV_1_RAW = ""
CSV_2_RAW = ""


In [ ]:
# Cell 3: Pretrained config
PRETRAIN_LOCAL_CKPT_RAW = ""
HF_PRETRAIN_REPO_ID = "Eempostor/F5-TTS-INDO-FINETUNE-V2"
HF_PRETRAIN_FILENAME = "f5_tts_indo_v2.pt"
HF_TOKEN_RAW = ""


In [ ]:
# Cell 4: Training config
TRAIN_CONFIG_NAME = "F5TTS_v1_Base_Mamba_Conservative.yaml"
TRAIN_BATCH_SIZE_PER_GPU = 8000
TRAIN_MAX_SAMPLES = 64
TRAIN_NUM_WORKERS = 4
TRAIN_EPOCHS = 10
TRAIN_LR = 1e-5
TRAIN_GRAD_ACCUMULATION_STEPS = 2
TRAIN_WARMUP_UPDATES = 500
ACCELERATE_NUM_PROCESSES = 1
ACCELERATE_MIXED_PRECISION = "fp16"


In [ ]:
# Cell 5: Helper dan path
import os
import shutil
import subprocess
import sys
from pathlib import Path


def clean_str(value: str) -> str:
    return (value or "").strip()


def run(cmd, cwd=None, env=None):
    printable = cmd if isinstance(cmd, str) else " ".join(str(x) for x in cmd)
    print(f"$ {printable}")
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=env, check=True, text=True)


KAGGLE_WORKING = Path("/kaggle/working")
KAGGLE_TEMP = Path("/kaggle/temp")
REPO_DIR = KAGGLE_WORKING / REPO_NAME
VENV_DIR = KAGGLE_TEMP / "f5tts-venv"
PYTHON_BIN = VENV_DIR / "bin/python"

DATASET_ROOT = Path(DATASET_ROOT_RAW).expanduser()
CSV_1 = Path(CSV_1_RAW).expanduser() if clean_str(CSV_1_RAW) else None
CSV_2 = Path(CSV_2_RAW).expanduser() if clean_str(CSV_2_RAW) else None
PRETRAIN_LOCAL_CKPT = Path(PRETRAIN_LOCAL_CKPT_RAW).expanduser() if clean_str(PRETRAIN_LOCAL_CKPT_RAW) else None
HF_TOKEN = clean_str(HF_TOKEN_RAW)

TRAIN_SAVE_DIR_REL = f"ckpts/F5TTS_v1_Base_Mamba_Conservative_vocos_pinyin_{DATASET_NAME}"
TRAIN_SAVE_DIR = REPO_DIR / TRAIN_SAVE_DIR_REL
PREPARED_DATASET_DIR = REPO_DIR / "data" / f"{DATASET_NAME}_pinyin"
MERGED_CSV = KAGGLE_WORKING / f"{DATASET_NAME}_merged.csv"

if PRETRAIN_LOCAL_CKPT is not None:
    PRETRAIN_TARGET_CKPT = TRAIN_SAVE_DIR / f"pretrained_{PRETRAIN_LOCAL_CKPT.name}"
else:
    PRETRAIN_TARGET_CKPT = TRAIN_SAVE_DIR / f"pretrained_{HF_PRETRAIN_FILENAME}"

print("REPO_DIR     :", REPO_DIR)
print("VENV_DIR     :", VENV_DIR)
print("DATASET_ROOT :", DATASET_ROOT)
print("MERGED_CSV   :", MERGED_CSV)
print("SAVE_DIR     :", TRAIN_SAVE_DIR)


In [ ]:
# Cell 6: Clone repo
KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
KAGGLE_TEMP.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    print("Repo sudah ada, skip clone:", REPO_DIR)
else:
    run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

run(["git", "-C", str(REPO_DIR), "branch", "--show-current"])


In [ ]:
# Cell 7: Cek file penting repo
required_files = [
    REPO_DIR / "src/f5_tts/train/train.py",
    REPO_DIR / "src/f5_tts/train/datasets/prepare_csv_wavs.py",
    REPO_DIR / "src/f5_tts/configs" / TRAIN_CONFIG_NAME,
    REPO_DIR / "src/f5_tts/model/hybrid_mamba.py",
]

missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Repo atau branch yang di-clone belum cocok untuk notebook ini:\n" + "\n".join(missing_files)
    )

print("Repo siap dipakai.")


In [ ]:
# Cell 8: Buat venv di /kaggle/temp
if VENV_DIR.exists():
    print("Venv sudah ada, skip buat ulang:", VENV_DIR)
else:
    run([sys.executable, "-m", "venv", str(VENV_DIR)])

run([str(PYTHON_BIN), "-m", "pip", "install", "--upgrade", "pip", "wheel", "setuptools<82"])


In [ ]:
# Cell 9: Install torch CUDA 12.8
run([
    str(PYTHON_BIN),
    "-m",
    "pip",
    "install",
    "--index-url",
    "https://download.pytorch.org/whl/cu128",
    "torch==2.8.0+cu128",
    "torchvision==0.23.0+cu128",
    "torchaudio==2.8.0+cu128",
])


In [ ]:
# Cell 10: Install repo dan dependency mamba
run([str(PYTHON_BIN), "-m", "pip", "install", "-e", str(REPO_DIR)], cwd=REPO_DIR)
run([str(PYTHON_BIN), "-m", "pip", "install", "ninja"])
run([str(PYTHON_BIN), "-m", "pip", "install", "mamba-ssm==2.3.1", "causal-conv1d>=1.4.0"])
run([str(PYTHON_BIN), "-m", "pip", "install", "-U", "huggingface_hub"])


In [ ]:
# Cell 11: Cek torch dan GPU
run([
    str(PYTHON_BIN),
    "-c",
    "import torch; "
    "print('torch =', torch.__version__); "
    "print('cuda =', torch.version.cuda); "
    "print('abi =', torch.compiled_with_cxx11_abi()); "
    "print('gpu_count =', torch.cuda.device_count()); "
    "assert torch.cuda.device_count() >= 1, 'GPU tidak terdeteksi'",
])


In [ ]:
# Cell 12: Cari metadata lokal
if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan: {DATASET_ROOT}")

if CSV_1 is None:
    found = sorted(DATASET_ROOT.glob("**/metadata.csv"))
    CSV_1 = found[0] if found else None

if CSV_1 is None:
    raise FileNotFoundError(
        "metadata.csv tidak ditemukan. Isi CSV_1_RAW kalau nama file metadata berbeda."
    )

if CSV_2 is not None and not CSV_2.exists():
    raise FileNotFoundError(f"CSV_2 tidak ditemukan: {CSV_2}")

print("DATASET_ROOT:", DATASET_ROOT)
print("CSV_1       :", CSV_1)
print("CSV_2       :", CSV_2)


In [ ]:
# Cell 13: Gabungkan metadata jadi satu CSV
import csv
from itertools import chain


def load_rows(csv_path: Path):
    rows = []
    with open(csv_path, "r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle, delimiter="|")
        first_row = next(reader, None)
        if first_row is None:
            return rows

        has_header = (
            len(first_row) >= 2
            and first_row[0].strip() == "audio_file"
            and first_row[1].strip() == "text"
        )
        iterator = reader if has_header else chain([first_row], reader)

        for row in iterator:
            if len(row) < 2:
                continue
            audio_file = row[0].strip()
            text = row[1].strip()
            if audio_file and text:
                rows.append({"audio_file": audio_file, "text": text})
    return rows


def resolve_audio_path(raw_path: str, csv_path: Path) -> Path:
    candidate = Path(raw_path).expanduser()
    if candidate.is_absolute() and candidate.exists():
        return candidate.resolve()

    for base in [csv_path.parent, DATASET_ROOT]:
        full_path = (base / raw_path).expanduser()
        if full_path.exists():
            return full_path.resolve()

    return (DATASET_ROOT / raw_path).expanduser().resolve()


merged_rows = []
seen_audio = set()

for csv_path in [CSV_1, CSV_2]:
    if csv_path is None:
        continue

    for row in load_rows(csv_path):
        audio_path = resolve_audio_path(row["audio_file"], csv_path)
        audio_key = str(audio_path)
        if audio_key in seen_audio:
            continue

        seen_audio.add(audio_key)
        merged_rows.append({
            "audio_file": audio_key,
            "text": row["text"],
        })

if not merged_rows:
    raise RuntimeError("Metadata kosong setelah digabung.")

with open(MERGED_CSV, "w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle, delimiter="|", quoting=csv.QUOTE_MINIMAL)
    writer.writerow(["audio_file", "text"])
    for row in merged_rows:
        writer.writerow([row["audio_file"], row["text"]])

print("Jumlah baris:", len(merged_rows))
print("Merged CSV  :", MERGED_CSV)
print("Contoh path :", merged_rows[0]["audio_file"])


In [ ]:
# Cell 14: Pastikan vocab ada
from huggingface_hub import hf_hub_download

vocab_path = REPO_DIR / "data" / "Emilia_ZH_EN_pinyin" / "vocab.txt"
vocab_path.parent.mkdir(parents=True, exist_ok=True)

if vocab_path.exists() and vocab_path.stat().st_size > 0:
    print("Vocab sudah ada:", vocab_path)
else:
    downloaded = hf_hub_download(repo_id="SWivid/F5-TTS", filename="F5TTS_v1_Base/vocab.txt")
    shutil.copy2(downloaded, vocab_path)
    print("Vocab berhasil disalin ke:", vocab_path)


In [ ]:
# Cell 15: Siapkan pretrained checkpoint
from huggingface_hub import hf_hub_download

TRAIN_SAVE_DIR.mkdir(parents=True, exist_ok=True)

if PRETRAIN_LOCAL_CKPT is not None:
    shutil.copy2(PRETRAIN_LOCAL_CKPT, PRETRAIN_TARGET_CKPT)
    print("Pakai checkpoint lokal:", PRETRAIN_TARGET_CKPT)
elif PRETRAIN_TARGET_CKPT.exists():
    print("Checkpoint pretrained sudah ada:", PRETRAIN_TARGET_CKPT)
else:
    download_kwargs = {
        "repo_id": HF_PRETRAIN_REPO_ID,
        "filename": HF_PRETRAIN_FILENAME,
    }
    if HF_TOKEN:
        download_kwargs["token"] = HF_TOKEN

    downloaded = hf_hub_download(**download_kwargs)
    shutil.copy2(downloaded, PRETRAIN_TARGET_CKPT)
    print("Checkpoint pretrained siap:", PRETRAIN_TARGET_CKPT)


In [ ]:
# Cell 16: Prepare dataset untuk repo ini
prepared_ok = (
    (PREPARED_DATASET_DIR / "raw.arrow").exists()
    and (PREPARED_DATASET_DIR / "duration.json").exists()
    and (PREPARED_DATASET_DIR / "vocab.txt").exists()
)

if prepared_ok:
    print("Prepared dataset sudah ada:", PREPARED_DATASET_DIR)
else:
    run([
        str(PYTHON_BIN),
        "src/f5_tts/train/datasets/prepare_csv_wavs.py",
        str(MERGED_CSV),
        str(PREPARED_DATASET_DIR),
        "--workers",
        str(TRAIN_NUM_WORKERS),
    ], cwd=REPO_DIR)

run(["ls", "-lah", str(PREPARED_DATASET_DIR)])


In [ ]:
# Cell 17: Jalankan fine-tune konservatif
train_env = os.environ.copy()
train_env["OMP_NUM_THREADS"] = str(TRAIN_NUM_WORKERS)
train_env["MKL_NUM_THREADS"] = str(TRAIN_NUM_WORKERS)
train_env["PYTHONFAULTHANDLER"] = "1"
train_env["WANDB_MODE"] = "disabled"

if (TRAIN_SAVE_DIR / "model_last.pt").exists():
    print("model_last.pt sudah ada. train.py akan auto-resume dari checkpoint itu.")
else:
    print("Training akan mulai dari pretrained:", PRETRAIN_TARGET_CKPT)

train_cmd = [
    str(PYTHON_BIN),
    "-m",
    "accelerate.commands.launch",
    f"--num_processes={ACCELERATE_NUM_PROCESSES}",
    f"--mixed_precision={ACCELERATE_MIXED_PRECISION}",
    "--dynamo_backend=no",
    "src/f5_tts/train/train.py",
    "--config-name",
    TRAIN_CONFIG_NAME,
    f"datasets.name={DATASET_NAME}",
    f"datasets.batch_size_per_gpu={TRAIN_BATCH_SIZE_PER_GPU}",
    f"datasets.max_samples={TRAIN_MAX_SAMPLES}",
    f"datasets.num_workers={TRAIN_NUM_WORKERS}",
    f"optim.epochs={TRAIN_EPOCHS}",
    f"optim.learning_rate={TRAIN_LR}",
    f"optim.num_warmup_updates={TRAIN_WARMUP_UPDATES}",
    f"optim.grad_accumulation_steps={TRAIN_GRAD_ACCUMULATION_STEPS}",
    "model.arch.attn_backend=torch",
    "model.arch.checkpoint_activations=True",
    f"ckpts.save_dir={TRAIN_SAVE_DIR_REL}",
    "ckpts.logger=null",
    "ckpts.log_samples=False",
    "ckpts.save_per_updates=5000",
    "ckpts.last_per_updates=500",
    "ckpts.keep_last_n_checkpoints=2",
]

run(train_cmd, cwd=REPO_DIR, env=train_env)


In [ ]:
# Cell 18: Cek hasil checkpoint
run(["ls", "-lah", str(TRAIN_SAVE_DIR)])

model_last = TRAIN_SAVE_DIR / "model_last.pt"
if model_last.exists():
    print("Training selesai. Checkpoint terakhir:", model_last)
else:
    print("model_last.pt belum ada. Cek log training di cell sebelumnya.")
